# 04 · State and memory

Every call to the agent in notebook 03 started from nothing. Real support conversations do not: a customer
asks where order A1001 is, then asks "when will it arrive exactly?" without repeating the number. In
LangGraph, memory comes from the **state** the graph carries and a **checkpointer** that saves it.

**You will learn**

- why an agent forgets between calls, and what a thread is
- how a checkpointer gives an agent per-conversation memory
- how to look inside a conversation's saved state and its history
- what changes when memory has to survive a restart, or span conversations

In [1]:
# Imports and setup shared by the cells below.
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

from agent_contracts.harness import verify_model
from agent_contracts.shipfast import tools as shipfast  # the shared ShipFast tools, which also log every call and refund
from lg_agents.llm import make_llm
from lg_agents.prompts import SPECIALIST  # notebook 03's instructions, now in lg_agents/prompts.py
from lg_agents.tools import shipfast_tools

verify_model()  # stops with the fix if Ollama or the shared model is missing
llm, tools = make_llm(), shipfast_tools()

# Two messages from one customer: the second only makes sense if the agent remembers the first.
FIRST, FOLLOW_UP = "Hi, where is my order A1003?", "When will it arrive exactly?"


def tool_calls(state):
    """The tool calls the agent made in a run, with their arguments."""
    return [f"{c['name']}({c['args']})" for m in state["messages"] for c in (getattr(m, "tool_calls", None) or [])]

## Without memory

The agent's state is a list of messages that exists only for one `invoke`. The customer asks about
A1003, the late espresso machine, then follows up. On its own, the follow-up has no order ID, so the
right move is to ask which order.

In [2]:
# An agent with no checkpointer: its state (the message list) exists only for the length of one invoke.
forgetful = create_agent(llm, tools, system_prompt=SPECIALIST)

# The customer's first message. We ignore the answer; what matters is whether the next run remembers it.
forgetful.invoke({"messages": [HumanMessage(FIRST)]})

{'messages': [HumanMessage(content='Hi, where is my order A1003?', additional_kwargs={}, response_metadata={}, id='ef59df18-99a9-408d-991b-6f4bfe3afb99'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3-agents', 'created_at': '2026-11-04T09:54:53.030718Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11713988792, 'load_duration': 896709, 'prompt_eval_count': 544, 'prompt_eval_duration': 7708789000, 'eval_count': 26, 'eval_duration': 3937478000, 'logprobs': None, 'model_name': 'qwen3-agents', 'model_provider': 'ollama'}, id='lc_run--01a1a5fb-7022-701c-8b8e-9c167fda1d37-0', tool_calls=[{'name': 'track_shipment', 'args': {'order_id': 'A1003'}, 'id': '1234f401-883d-4e1b-8b93-9d499950622c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 544, 'output_tokens': 26, 'total_tokens': 570}),
  ToolMessage(content='{"order_id": "A1003", "status": "delayed", "carrier": "LBC", "eta": "2026-10-24", "delivered_on": null, "trackin

Now send the follow-up as a separate `invoke`. Nothing from the first run is passed along, so the agent sees only the follow-up.

In [3]:
# The follow-up arrives in a new run, so the state starts empty again.
state = forgetful.invoke({"messages": [HumanMessage(FOLLOW_UP)]})

calls = tool_calls(state)  # every tool the agent called during this run
print("tool calls:", ", ".join(calls) or "none")
print("reply:     ", state["messages"][-1].content.strip())  # the last message is the agent's answer

tool calls: track_shipment({'order_id': 'order_12345'})
reply:      I apologize for the inconvenience. It seems there was an issue retrieving the delivery details for your order. Could you please provide the order ID again so I can look it up for you?


Look at the tool calls. Nothing in that second run mentions an order, yet when we ran it, the agent
looked one up anyway, with an ID it made up (`order_12345`). The shared tool answered `order_not_found`,
and only then did the agent ask the customer for the ID. The tool descriptions say never to guess an
ID; a small model does not always listen, and the tool's error is what kept the answer honest.

An earlier version of the shared `lookup_order` description gave "A1001" as an example ID. The model
copied it, and since A1001 is a real order it answered confidently about the wrong order. That is why the
description no longer contains a real ID: a guess should fail loudly, not succeed quietly.

## A checkpointer and a thread

Give the agent a **checkpointer** and LangGraph saves the state after every step. Each conversation gets
a **thread id** in the config; calling again with the same id continues from the saved state, so the new
message is appended to the old ones. `InMemorySaver` keeps the checkpoints in this Python process.

In [4]:
from langgraph.checkpoint.memory import InMemorySaver

# The checkpointer saves the state after every step; InMemorySaver keeps it in this Python process.
agent = create_agent(llm, tools, system_prompt=SPECIALIST, checkpointer=InMemorySaver())

# The thread id names one conversation. Pass the same config on every call to continue it.
maria = {"configurable": {"thread_id": "maria"}}

Now send both messages with the same config. Each call is given only the new message; the checkpointer supplies the earlier ones.

In [5]:
shipfast.reset()  # clear the tool-call log, so only this conversation shows up
for text in (FIRST, FOLLOW_UP):
    reply = agent.invoke({"messages": [HumanMessage(text)]}, maria)["messages"][-1]  # maria: continue her thread
    print(f"customer: {text}\nagent:    {reply.content.strip()}\n")
print("tools called:", ", ".join(shipfast.tool_calls()) or "none")

customer: Hi, where is my order A1003?
agent:    Your order A1003 is currently delayed and is expected to be delivered by LBC on October 24, 2026. The latest update shows it is held at the Davao hub due to a backlog. We apologize for the inconvenience and are working to resolve this as soon as possible.



customer: When will it arrive exactly?
agent:    Your order A1003 is expected to arrive by LBC on October 24, 2026. The exact delivery date may vary slightly depending on the backlog at the Davao hub. We apologize for the delay and are working to resolve this as quickly as possible.

tools called: track_shipment, track_shipment


This time the follow-up is answered about A1003, the order the customer actually asked about. A
different thread id is a different conversation, with nothing in it, so the same follow-up there is back
to the no-memory case:

In [6]:
# A different thread id is a different conversation: Maria's saved messages are not visible here.
someone_else = {"configurable": {"thread_id": "someone-else"}}
state = agent.invoke({"messages": [HumanMessage(FOLLOW_UP)]}, someone_else)

calls = tool_calls(state)
print("tool calls:", ", ".join(calls) or "none")
print("reply:     ", state["messages"][-1].content.strip())

tool calls: track_shipment({'order_id': 'order_12345'})
reply:      I apologize for the inconvenience. It seems there was an issue retrieving the delivery details for your order. Could you please provide the order ID again so I can look it up for you?


## Looking inside a thread

`get_state` returns the latest checkpoint: the state values (here, every message so far) and which node
would run next (none, since the run finished).

In [7]:
snapshot = agent.get_state(maria)  # the latest checkpoint of Maria's thread
messages = snapshot.values["messages"]  # the state values: every message so far

print("messages:   ", len(messages))
print(f"  {'type':6} content")
for m in messages:
    calls = getattr(m, "tool_calls", None)  # only the agent's tool-request messages have any
    # a tool request shows the tool names; every other message shows its first 80 characters of text
    print(f"  {m.type:6}", ", ".join(c["name"] for c in calls) if calls else m.content.strip()[:80])
print("next node:  ", snapshot.next or "none (the run finished)")

messages:    8
  type   content
  human  Hi, where is my order A1003?
  ai     track_shipment
  tool   {"order_id": "A1003", "status": "delayed", "carrier": "LBC", "eta": "2026-10-24"
  ai     Your order A1003 is currently delayed and is expected to be delivered by LBC on 
  human  When will it arrive exactly?
  ai     track_shipment
  tool   {"order_id": "A1003", "status": "delayed", "carrier": "LBC", "eta": "2026-10-24"
  ai     Your order A1003 is expected to arrive by LBC on October 24, 2026. The exact del
next node:   none (the run finished)


`get_state_history` lists every checkpoint, newest first. LangGraph can replay or branch from any of them, which is useful for debugging. Here you only count them.

In [8]:
history = list(agent.get_state_history(maria))  # one checkpoint per step of the conversation, newest first
print("checkpoints:", len(history))

checkpoints: 10


## Memory that changes the outcome

Scenario S4 needs two turns. The vase from A1004 arrived broken; policy P3 requires a photo before a
refund. The first turn should ask for it, and the second, which never mentions the order, should refund
the $60 total. Only memory connects the two.

In [9]:
shipfast.reset()
s4 = {"configurable": {"thread_id": "S4"}}  # a fresh thread for this scenario
for text in ("My vase set from order A1004 arrived broken.", "Here is the photo: photo_A1004.jpg"):
    reply = agent.invoke({"messages": [HumanMessage(text)]}, s4)["messages"][-1]
    refunds = [f"{r['order_id']} ${r['amount']}" for r in shipfast.refunds()]  # refunds actually issued so far
    print(f"customer: {text}")
    print(f"agent:    {reply.content.strip()[:220]}")
    print(f"ledger:   {', '.join(refunds) or 'no refund'}\n")

customer: My vase set from order A1004 arrived broken.
agent:    The vase set from your order A1004 arrived broken. According to our policy, you must provide a photo of the damage within 7 days of the delivery date. Since the delivery was on October 30, 2026, and today is November 1, 
ledger:   no refund



customer: Here is the photo: photo_A1004.jpg
agent:    Your full refund of $60.00 for the broken vase set in order A1004 has been processed. You should see the refund reflected in your account within a few business days. If you have any further questions, feel free to ask!
ledger:   A1004 $60.0



## When memory has to last

- **Surviving a restart.** `InMemorySaver` forgets everything when the process ends. LangGraph has
  database checkpointers with the same interface (for SQLite, the `langgraph-checkpoint-sqlite` package;
  for Postgres, `langgraph-checkpoint-postgres`). Swapping one in is the only change; the threads then
  survive restarts and can be resumed from another process. This tutorial stays in memory.
- **Across conversations.** A checkpointer is per thread. Facts that should follow a customer into a new
  conversation, such as a preferred name, go in a **store** (`langgraph.store.memory.InMemoryStore`, or a
  database store), passed to `create_agent(store=...)` and read or written by tools.
- **Long threads.** Every message is sent back to the model on every turn, so long threads grow the
  prompt until they hit the 16K context. `SummarizationMiddleware` can compress older turns.

## How the others do this

- **CrewAI** turns memory on per crew (`Crew(memory=True)`), with short-term, long-term and entity
  memory. It needs an embedding model, OpenAI's by default; keeping it local means configuring an Ollama
  embedder, which the CrewAI tutorial does.
- **Strands** keeps the conversation in `agent.messages`, so calling the same `Agent` again continues it.
  A session manager such as `FileSessionManager` saves and restores it across restarts.

## Takeaways

- The agent's state lives for one run unless a checkpointer saves it.
- A thread id names a conversation; the same id continues it, a new id starts fresh.
- Checkpoints can be inspected, replayed and, with a database checkpointer, kept across restarts.
- Next: **05 · Orchestration**, where you draw the graph yourself.